# Inference Multi-Character Aksara Lontara
## Fine-Tuned BLIP – Deteksi 1 sampai 4 Karakter per Gambar

### Penjelasan Pendekatan

Model BLIP yang telah dilatih pada gambar **tunggal** (1 karakter per gambar) sudah menunjukkan akurasi **100%**.
Notebook ini memanfaatkan model tersebut untuk mengenali **1 hingga 4 karakter** dalam satu gambar.

**Alur kerja:**

1. **Preprocessing** – Grayscale → Gaussian blur → adaptive threshold → morphological cleanup.
2. **Vertical Projection Segmentation** – Menghitung jumlah piksel hitam per kolom → menemukan gap (lembah) antar karakter → membelah gambar di titik gap. Pendekatan ini secara otomatis mengelompokkan titik/tanda dengan badan karakter di bawahnya karena berada di kolom yang sama.
3. **Tight Bounding Box** – Setiap region karakter di-crop dengan bounding box ketat + padding.
4. **Inference Individual** – Setiap potongan di-resize ke 384×384 dan di-caption oleh BLIP (identik dengan `inference_local.ipynb`).
5. **Penggabungan Caption** – Hasil caption per karakter digabung menjadi satu deskripsi utuh.

**Mengapa Vertical Projection?**
Karakter Lontara sering memiliki **titik terpisah** di atas/bawah badan karakter. Contour detection biasa akan mendeteksi titik sebagai contour terpisah dan bisa salah merge. Vertical projection menghindari masalah ini karena titik dan badan karakter berada di kolom horizontal yang sama.

**Catatan Penting:**
- Aksara Lontara: **19 karakter dasar × 5 pola = 95 karakter** total.
- Pastikan karakter memiliki **jarak horizontal cukup** antar karakter.
- Maksimal **4 karakter** per gambar.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP, IMPORT & MOUNT GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════

import os
import re
import json
import cv2
import math
import torch
import numpy as np
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import matplotlib.pyplot as plt

# Mount Google Drive (khusus Colab)
from google.colab import drive
drive.mount('/content/drive')

# ─── Konfigurasi ───
DRIVE_DIR  = '/content/drive/MyDrive/ImageCaptioning'
MODEL_DIR  = f'{DRIVE_DIR}/model/blip_lontara/best_model'
TEST_DIR   = f'{DRIVE_DIR}/datatest'           # Folder gambar uji
MAX_LENGTH = 64
NUM_BEAMS  = 4
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device aktif: {DEVICE}')
print(f'Model path  : {MODEL_DIR}')
print(f'Test path   : {TEST_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: LOAD MODEL BLIP HASIL FINE-TUNING
# ═══════════════════════════════════════════════════════════════

assert os.path.exists(MODEL_DIR), f'Model tidak ditemukan di {MODEL_DIR}! Pastikan hasil training sudah di-copy ke Drive.'

print(f'Memuat model dari: {MODEL_DIR}')
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

print('✅ Model BLIP siap digunakan!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: FUNGSI SEGMENTASI & INFERENCE MULTI-KARAKTER (1–4)
# ═══════════════════════════════════════════════════════════════
# Vertical Projection Segmentation + Canvas = ukuran gambar asli
# Crop karakter ditempatkan pada canvas berukuran SAMA dengan gambar asli,
# sehingga pipeline processor BLIP identik dengan inference 1 karakter.

MAX_CHARS = 4   # Maksimal karakter per gambar

# ─── Preprocessing ───

def preprocess_binary(image_path):
    """Konversi gambar ke binary mask (karakter = putih, bg = hitam)."""
    img_cv = cv2.imread(image_path)
    if img_cv is None:
        raise ValueError(f'Gagal membaca gambar: {image_path}')

    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=25, C=10
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=1)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    return img_cv, gray, thresh


# ─── Vertical Projection Segmentation ───

def find_split_points(projection, min_gap_width=3):
    """Cari gap (kolom berturut-turut dengan projection == 0)."""
    gaps = []
    in_gap = False
    gap_start = 0

    for i, val in enumerate(projection):
        if val == 0:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_width = i - gap_start
                if gap_width >= min_gap_width:
                    gaps.append((gap_start, i, gap_width))
                in_gap = False

    if in_gap:
        gap_width = len(projection) - gap_start
        if gap_width >= min_gap_width:
            gaps.append((gap_start, len(projection), gap_width))

    return gaps


def segment_by_projection(image_path, min_gap_ratio=0.02, debug=False):
    """
    Segmentasi karakter menggunakan Vertical Projection Profile.

    Returns:
        boxes: list of (x1, y1, x2, y2) — bounding box ketat per karakter
        img_cv, thresh, projection: untuk debug
    """
    img_cv, gray, thresh = preprocess_binary(image_path)
    h_img, w_img = gray.shape

    projection = np.sum(thresh > 0, axis=0)
    min_gap_px = max(2, int(w_img * min_gap_ratio))
    gaps = find_split_points(projection, min_gap_width=min_gap_px)

    if debug:
        print(f'  Image size: {w_img}x{h_img}')
        print(f'  Min gap width: {min_gap_px}px')
        print(f'  Gaps found: {len(gaps)}')
        for i, (gs, ge, gw) in enumerate(gaps):
            print(f'    Gap {i+1}: col {gs}–{ge} (width={gw})')

    content_cols = np.where(projection > 0)[0]
    if len(content_cols) == 0:
        return [], img_cv, thresh, projection

    col_start = content_cols[0]
    col_end = content_cols[-1] + 1

    inner_gaps = [(gs, ge, gw) for gs, ge, gw in gaps
                  if gs > col_start and ge < col_end]

    if debug:
        print(f'  Content range: col {col_start}–{col_end}')
        print(f'  Inner gaps (between chars): {len(inner_gaps)}')

    if len(inner_gaps) > MAX_CHARS - 1:
        inner_gaps = sorted(inner_gaps, key=lambda g: g[2], reverse=True)[:MAX_CHARS - 1]
        inner_gaps = sorted(inner_gaps, key=lambda g: g[0])

    split_cols = [col_start]
    for gs, ge, gw in inner_gaps:
        mid = (gs + ge) // 2
        split_cols.append(mid)
    split_cols.append(col_end)

    raw_boxes = []
    for r in range(len(split_cols) - 1):
        x_left = split_cols[r]
        x_right = split_cols[r + 1]

        region_strip = thresh[:, x_left:x_right]
        row_proj = np.sum(region_strip > 0, axis=1)
        active_rows = np.where(row_proj > 0)[0]

        if len(active_rows) == 0:
            continue

        y_top = active_rows[0]
        y_bot = active_rows[-1] + 1

        # Bounding box ketat (hanya 2px safety margin)
        x1 = max(0, x_left - 2)
        y1 = max(0, y_top - 2)
        x2 = min(w_img, x_right + 2)
        y2 = min(h_img, y_bot + 2)

        if (x2 - x1) > 5 and (y2 - y1) > 5:
            raw_boxes.append((x1, y1, x2, y2))

    if debug:
        print(f'  → Karakter terdeteksi: {len(raw_boxes)}')
        for i, (x1, y1, x2, y2) in enumerate(raw_boxes):
            print(f'    Box {i+1}: x={x1}–{x2} (w={x2-x1}), y={y1}–{y2} (h={y2-y1})')

    return raw_boxes, img_cv, thresh, projection


# ─── Crop + Canvas (ukuran gambar asli) ───

def crop_character_on_canvas(pil_img, box, scale=1.0):
    """
    Crop karakter lalu tempatkan di tengah canvas putih.

    KUNCI AKURASI:
    Canvas berukuran SAMA dengan gambar asli (misal 118×118),
    sehingga saat processor BLIP me-resize ke 384×384,
    pipeline-nya IDENTIK dengan inference 1 karakter.

    Args:
        pil_img: gambar PIL asli (multi-karakter)
        box: (x1, y1, x2, y2) bounding box ketat
        scale: faktor skala karakter pada canvas.
               1.0 = ukuran piksel asli.
               >1.0 = perbesar karakter (jika terlalu kecil di canvas).
               Rekomendasi: mulai 1.0, naikkan jika karakter terlalu kecil.
    """
    x1, y1, x2, y2 = box
    cropped = pil_img.crop((x1, y1, x2, y2))
    cw, ch = cropped.size

    # Canvas = ukuran gambar asli (persegi)
    orig_w, orig_h = pil_img.size
    canvas_dim = max(orig_w, orig_h)

    # Scale karakter jika diperlukan
    if scale != 1.0:
        new_cw = max(1, int(cw * scale))
        new_ch = max(1, int(ch * scale))
        # Jangan melebihi canvas
        new_cw = min(new_cw, canvas_dim - 4)
        new_ch = min(new_ch, canvas_dim - 4)
        cropped = cropped.resize((new_cw, new_ch), Image.LANCZOS)
        cw, ch = new_cw, new_ch

    # Buat canvas putih, paste karakter di tengah
    canvas = Image.new('RGB', (canvas_dim, canvas_dim), (255, 255, 255))
    paste_x = (canvas_dim - cw) // 2
    paste_y = (canvas_dim - ch) // 2
    canvas.paste(cropped, (paste_x, paste_y))

    return canvas


# ─── Inference (identik dengan inference_local.ipynb) ───

def predict_single(pil_img):
    """
    Captioning 1 gambar menggunakan BLIP.
    Parameter generate IDENTIK dengan inference_local.ipynb.
    """
    inputs = processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=2
        )
    return processor.decode(output[0], skip_special_tokens=True)


# ─── Gabung Caption ───

def extract_char_name(caption):
    """Ekstrak nama karakter dari caption model."""
    patterns = [
        r'yaitu karakter\s+(\w+)',
        r'adalah\s+(\w+)\s+dengan',
        r'merupakan karakter\s+(\w+)',
        r'karakter\s+(\w+)\s+memiliki',
        r'karakter\s+(\w+)\s+dengan',
    ]
    for pat in patterns:
        m = re.search(pat, caption, re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def combine_captions(captions):
    """Gabungkan caption individual menjadi satu deskripsi utuh (1–4 karakter)."""
    n = len(captions)
    if n == 0:
        return 'Tidak ada karakter yang terdeteksi.'
    if n == 1:
        return captions[0]

    names = [extract_char_name(c) for c in captions]
    valid_names = [nm for nm in names if nm is not None]

    if len(valid_names) == n:
        if n == 2:
            nama_str = f'{valid_names[0]} dan {valid_names[1]}'
        else:
            nama_str = ', '.join(valid_names[:-1]) + f', dan {valid_names[-1]}'
        intro = f'Gambar ini terdiri dari {n} karakter lontara yaitu karakter {nama_str}.'
    else:
        intro = f'Gambar ini terdiri dari {n} karakter lontara.'

    details = []
    for i, cap in enumerate(captions, 1):
        name = names[i-1] if names[i-1] else '?'
        clean = re.sub(r'^Gambar ini terdiri dari \d+ karakter lontara[^.]*\.?\s*', '', cap).strip()
        if not clean:
            clean = cap
        details.append(f'Karakter ke-{i} ({name}): {clean}')

    return intro + ' ' + ' '.join(details)


# ─── Entry Point ───

# Parameter scale: tuning utama jika akurasi masih kurang.
# - 1.0 = ukuran piksel asli (karakter kecil di canvas)
# - 1.5 = perbesar 1.5x (jika karakter terlalu kecil vs training)
# - 2.0 = perbesar 2x
CHAR_SCALE = 1.0

def predict_multi(image_path, debug=False):
    """
    Segmentasi → crop pada canvas (ukuran gambar asli) → inference → gabung caption.

    Returns:
        (caption_gabungan, list_img_canvas, list_caption, list_boxes, projection)
    """
    pil_img = Image.open(image_path).convert('RGB')

    boxes, img_cv, thresh, projection = segment_by_projection(
        image_path, min_gap_ratio=0.02, debug=debug
    )

    if len(boxes) <= 1:
        cap = predict_single(pil_img)
        return cap, [pil_img], [cap], [], projection

    char_imgs = []
    char_caps = []
    for box in boxes:
        cimg = crop_character_on_canvas(pil_img, box, scale=CHAR_SCALE)
        char_imgs.append(cimg)
        char_caps.append(predict_single(cimg))

    combined = combine_captions(char_caps)
    return combined, char_imgs, char_caps, boxes, projection

print(f'✅ Segmentasi + Canvas (ukuran asli, scale={CHAR_SCALE}) siap!')
print(f'   Jika akurasi kurang, ubah CHAR_SCALE (1.0 → 1.5 → 2.0)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: DEMO INFERENCE (SATU GAMBAR — 1 SAMPAI 4 KARAKTER)
# ═══════════════════════════════════════════════════════════════

# Ganti dengan nama file gambar uji Anda
TEST_IMAGE = f'{TEST_DIR}/contoh_multi.png'   # <-- sesuaikan nama file

if not os.path.exists(TEST_IMAGE):
    print(f'⚠️ Gambar tidak ditemukan: {TEST_IMAGE}')
    print('Pastikan Anda sudah meng-upload gambar ke folder datatest di Google Drive.')
else:
    combined, char_imgs, char_caps, boxes, projection = predict_multi(TEST_IMAGE, debug=True)

    n = len(char_imgs)

    print('=' * 60)
    print(f'Jumlah karakter terdeteksi: {n}')
    print('=' * 60)
    print('📌 CAPTION GABUNGAN:')
    print(combined)
    print('=' * 60)
    print('📋 DETAIL PER KARAKTER:')
    for i, cap in enumerate(char_caps, 1):
        print(f'  [{i}] {cap}')
    print('=' * 60)

    # ─── Visualisasi Debug: Threshold + Vertical Projection ───
    img_cv_dbg, _, thresh_dbg = preprocess_binary(TEST_IMAGE)
    fig_dbg, axes_dbg = plt.subplots(1, 3, figsize=(18, 4))

    # Gambar asli
    axes_dbg[0].imshow(cv2.cvtColor(img_cv_dbg, cv2.COLOR_BGR2RGB))
    axes_dbg[0].set_title('Gambar Asli', fontweight='bold')
    axes_dbg[0].axis('off')

    # Binary mask
    axes_dbg[1].imshow(thresh_dbg, cmap='gray')
    axes_dbg[1].set_title('Binary Mask (Threshold)', fontweight='bold')
    axes_dbg[1].axis('off')

    # Vertical projection profile
    axes_dbg[2].fill_between(range(len(projection)), projection, alpha=0.7, color='steelblue')
    axes_dbg[2].set_title('Vertical Projection Profile', fontweight='bold')
    axes_dbg[2].set_xlabel('Kolom (x)')
    axes_dbg[2].set_ylabel('Jumlah piksel hitam')
    # Tandai split points
    for box in boxes:
        axes_dbg[2].axvline(x=box[0], color='red', linestyle='--', alpha=0.5, linewidth=1)
        axes_dbg[2].axvline(x=box[2], color='red', linestyle='--', alpha=0.5, linewidth=1)

    plt.suptitle('Debug Segmentasi', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ─── Visualisasi Hasil ───
    if n == 1 and len(boxes) == 0:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(Image.open(TEST_IMAGE).convert('RGB'))
        axes[0].set_title('Gambar Input', fontsize=11, fontweight='bold')
        axes[0].axis('off')
        axes[1].text(0.5, 0.5, char_caps[0], wrap=True,
                     ha='center', va='center', fontsize=10,
                     transform=axes[1].transAxes)
        axes[1].set_title('Caption', fontsize=11, fontweight='bold')
        axes[1].axis('off')
    else:
        total_cols = n + 1  # kolom 0 = input, kolom 1..n = karakter
        fig, axes = plt.subplots(2, total_cols, figsize=(5 * total_cols, 10))

        # Baris 1, kolom 0: Gambar input + bounding boxes
        img_input = Image.open(TEST_IMAGE).convert('RGB')
        img_draw = np.array(img_input).copy()
        colors = [(255, 0, 0), (0, 180, 0), (0, 0, 255), (255, 165, 0)]
        for idx_b, box in enumerate(boxes):
            x1, y1, x2, y2 = box
            clr = colors[idx_b % len(colors)]
            cv2.rectangle(img_draw, (x1, y1), (x2, y2), clr, 3)
            cv2.putText(img_draw, str(idx_b + 1), (x1 + 5, y1 + 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, clr, 2)
        axes[0, 0].imshow(img_draw)
        axes[0, 0].set_title('Input + Bounding Box', fontsize=10, fontweight='bold')
        axes[0, 0].axis('off')

        # Baris 1, kolom 1..n: crop per karakter
        for i in range(n):
            axes[0, i + 1].imshow(char_imgs[i])
            axes[0, i + 1].set_title(f'Karakter {i + 1}', fontsize=10, fontweight='bold')
            axes[0, i + 1].axis('off')

        # Baris 2, kolom 0: Caption gabungan
        axes[1, 0].text(0.5, 0.5, combined, wrap=True,
                        ha='center', va='center', fontsize=9,
                        transform=axes[1, 0].transAxes,
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow'))
        axes[1, 0].set_title('Caption Gabungan', fontsize=10, fontweight='bold')
        axes[1, 0].axis('off')

        # Baris 2, kolom 1..n: caption individual
        for i, cap in enumerate(char_caps):
            axes[1, i + 1].text(0.5, 0.5, cap, wrap=True,
                               ha='center', va='center', fontsize=9,
                               transform=axes[1, i + 1].transAxes,
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcyan'))
            axes[1, i + 1].set_title(f'Caption Karakter {i + 1}', fontsize=10)
            axes[1, i + 1].axis('off')

    plt.suptitle(f'Inference: {n} karakter terdeteksi', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: BATCH INFERENCE (SELURUH FOLDER DATATEST)
# ═══════════════════════════════════════════════════════════════

VALID_EXT = ('.png', '.jpg', '.jpeg')

if not os.path.exists(TEST_DIR):
    print(f'⚠️ Folder {TEST_DIR} tidak ditemukan. Buat folder datatest di Google Drive Anda.')
else:
    images = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(VALID_EXT)])
    print(f'Ditemukan {len(images)} gambar di folder datatest\n')

    results = []
    for img_name in images:
        img_path = os.path.join(TEST_DIR, img_name)
        try:
            cap, char_imgs, char_caps, boxes, _ = predict_multi(img_path)
            n_chars = len(char_imgs)
            results.append({
                'gambar': img_name,
                'jumlah_karakter': n_chars,
                'caption_gabungan': cap,
                'caption_per_karakter': char_caps
            })
            print(f'{img_name} ({n_chars} char) → {cap}')
        except Exception as e:
            print(f'{img_name} → ERROR: {e}')

    # Simpan hasil ke Drive
    out_dir = f'{DRIVE_DIR}/hasil_inference_multi'
    os.makedirs(out_dir, exist_ok=True)
    out_file = f'{out_dir}/hasil_multi.json'
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'\n✅ Hasil disimpan di: {out_file}')

    # ─── Visualisasi grid (maks 12 gambar) ───
    show = results[:12]
    cols = min(4, len(show))
    rows = math.ceil(len(show) / cols) if cols > 0 else 1
    if len(show) > 0:
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
        if rows * cols == 1:
            axes = [axes]
        else:
            axes = np.array(axes).flatten()
        for idx, item in enumerate(show):
            img = Image.open(os.path.join(TEST_DIR, item['gambar']))
            axes[idx].imshow(img)
            title = f"{item['gambar']}\n({item['jumlah_karakter']} char)"
            axes[idx].set_title(title, fontsize=8)
            axes[idx].axis('off')
        for idx in range(len(show), len(axes)):
            axes[idx].axis('off')
        plt.tight_layout()
        plt.show()

    print(f'\n✅ Selesai! Total {len(results)} gambar diproses.')